# AutoEdit AI Render Deployment Validation

This notebook tests that all containers and services build, start, and work as expected for Render-ready deployment.


In [ ]:
# 1. Import Required Libraries
import subprocess
import requests
import time
import json
from pathlib import Path


In [ ]:
# 2. Main Functionality Implementation

def run_command(cmd, cwd=".", timeout=300):
    """Run a terminal command and return (success, output)."""
    try:
        result = subprocess.run(cmd, shell=True, cwd=cwd, timeout=timeout, capture_output=True, text=True)
        return result.returncode == 0, result.stdout + result.stderr
    except Exception as e:
        return False, str(e)

def build_images():
    return run_command("docker compose build")

def up_stack():
    # --wait flag not present so use script sleep below
    return run_command("docker compose up -d")

def check_container_health(service, max_wait=60):
    for _ in range(max_wait):
        healthy, out = run_command(f"docker inspect -f '{{{{.State.Health.Status}}}}' {service}")
        if healthy and "healthy" in out:
            return True
        time.sleep(2)
    return False

def get_api_health():
    try:
        r = requests.get("http://localhost:8000/health", timeout=10)
        return r.ok, r.json()
    except Exception as ex:
        return False, str(ex)


In [ ]:
# 3. Run the Notebook Code

# Build docker images
build_ok, build_log = build_images()
print("Docker Compose Build Success?", build_ok)
print(build_log[-1000:])  # tail

# Bring up docker compose stack
up_ok, up_log = up_stack()
print("Docker Compose Up Success?", up_ok)
print(up_log[-1000:])

# Check status of containers (API, worker, web, redis, postgres)
api_healthy = check_container_health("autoedit-ai-api-1")
worker_healthy = check_container_health("autoedit-ai-worker-1")
youtube_worker_healthy = check_container_health("autoedit-ai-youtube-worker-1")
web_healthy = check_container_health("autoedit-ai-web-1")
redis_healthy = check_container_health("autoedit-ai-redis-1")
postgres_healthy = check_container_health("autoedit-ai-postgres-1")

print("API healthy?", api_healthy)
print("Worker healthy?", worker_healthy)
print("YouTube Worker healthy?", youtube_worker_healthy)
print("Web healthy?", web_healthy)
print("Redis healthy?", redis_healthy)
print("Postgres healthy?", postgres_healthy)


In [ ]:
# 4. Display Output

# Fetch API health endpoint and show results
api_ok, api_health = get_api_health()
print(f"API /health endpoint success? {api_ok}")
print("API /health endpoint result:")
print(json.dumps(api_health, indent=2) if api_ok else api_health)

# Here you would run a real video pipeline/test job, and verify the end-to-end flow (not shown in this basic validation notebook).
